# 002 Build a Minimal Weather Skill

这是第二课：从零新建一份最小 Skill。

学习目标：

1. 学会从一个简单需求出发，判断最小 Skill 应该长什么样
2. 学会先写最小可用的 `SKILL.md`
3. 学会什么时候只需要 `SKILL.md`，什么时候再补 `agents/openai.yaml`
4. 用一个真实的天气查询 Skill 建立正式开发直觉

这次我们不用复杂任务，而是故意选一个边界非常清楚的例子：

- 天气查询 Skill


## 先明确这节课的目标

这节课不是教你做天气 API，也不是教你写复杂工具链。

目标只有一个：

- 学会从零落一份最小 Skill

所以我们这次刻意控制范围：

- 只做天气查询
- 只保留最必要的工作流
- 不先加 `references/`
- 不先加 `scripts/`

这样你才能看清最小 Skill 到底是什么。


## 先把需求说成人话

假设你现在有这样一个需求：

“当用户问天气时，希望智能体能按固定方式处理，不要答得太散。”

这里真正要沉淀的不是天气数据本身，而是处理方式：

1. 先确认地点
2. 再查天气
3. 再用简洁方式回答

这已经是一个很适合做成 Skill 的小工作流。


In [1]:
weather_skill_goal = {
    'task': '处理天气查询',
    'why_skill': [
        '有稳定 workflow',
        '回答风格可以标准化',
        '以后可能被重复使用',
    ],
    'not_in_scope': [
        '自己实现天气 API',
        '复杂的旅行建议',
        '长篇气候科普',
    ],
}

from pprint import pprint
pprint(weather_skill_goal)


{'not_in_scope': ['自己实现天气 API', '复杂的旅行建议', '长篇气候科普'],
 'task': '处理天气查询',
 'why_skill': ['有稳定 workflow', '回答风格可以标准化', '以后可能被重复使用']}


## 正式开发时，先决定最小边界

很多人一上来就会想建这些东西：

- `references/`
- `scripts/`
- `assets/`
- 一堆测试和模板

这一步太早了。

对于这节课这个天气 Skill，最小边界是：

- `SKILL.md`
- `agents/openai.yaml`

也就是说，先把“这类任务该怎么做”说清楚，再考虑要不要增加额外资源。


## 看看真实目录

这份最小天气 Skill 我已经直接落到仓库里了。

路径是：

- `.agents/skills/weather-query-assistant/`


In [2]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


## 打印目录树

注意这里很关键：

- 没有 `references/`
- 没有 `scripts/`
- 也没有 `assets/`

这不是缺失，而是有意保持最小结构。


In [3]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
└── SKILL.md


## 先读 `SKILL.md`

正式开发时，第一版 Skill 最重要的文件永远是 `SKILL.md`。

因为这里定义的是：

- 什么时候用这份 Skill
- 这类请求按什么步骤处理
- 回答风格有什么约束


In [4]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Use `wttr.in` as the primary source.
4. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
5. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
6. Do not guess when weather data is unavailable.

## wttr.in (primary)

Quick one-liner:

```bash
curl -s "wttr.in/London?format=3"
# Output: London: ⛅️ +8°C
```

Compact format:

```bash
curl -s "wttr.in/London?format=%l:+%c+%t+%h+%w"
# Out

## 读完之后，你要能看出这份 Skill 的最小骨架

最小 `SKILL.md` 其实只要解决 3 件事：

1. `name`
2. `description`
3. workflow

这三件事不清楚，后面加多少目录都没意义。


In [5]:
minimal_skill_parts = {
    'frontmatter_required': ['name', 'description'],
    'body_required': ['when to use', 'workflow'],
    'body_optional': ['style', 'references', 'scripts'],
}

pprint(minimal_skill_parts)


{'body_optional': ['style', 'references', 'scripts'],
 'body_required': ['when to use', 'workflow'],
 'frontmatter_required': ['name', 'description']}


## 为什么这份天气 Skill 现在还不需要 `references/`

因为它的规则还很短：

- 确认地点
- 查天气
- 简洁回答
- 缺数据就明确说明

这些内容完全放得下，也适合先放在 `SKILL.md` 里。

只有当你后面开始增加复杂规则时，比如：

- 多种地点格式
- 多语言回答约定
- 天气响应字段映射
- 特殊城市名规范

这时候才值得拆出 `references/`。


In [6]:
reasons_no_references_yet = [
    'workflow short',
    'no large domain docs yet',
    'no multiple variants yet',
]

pprint(reasons_no_references_yet)


['workflow short', 'no large domain docs yet', 'no multiple variants yet']


## 为什么这份天气 Skill 现在也不需要 `scripts/`

因为它还没有重复性的本地动作。

目前这份 Skill 的职责只是告诉智能体：

- 如何处理天气问题
- 如何组织回答

它还不需要：

- 本地校验脚本
- 模板生成脚本
- 批量处理工具

如果以后你发现某个动作开始重复出现，例如：

- 统一校验天气返回格式
- 标准化地点名称

那时再加 `scripts/` 更合理。


In [7]:
reasons_no_scripts_yet = [
    'no repeated local action yet',
    'no validation helper needed yet',
    'no deterministic transformation step yet',
]

pprint(reasons_no_scripts_yet)


['no repeated local action yet',
 'no validation helper needed yet',
 'no deterministic transformation step yet']


## 再看 `agents/openai.yaml`

虽然这不是必需文件，但在正式开发里通常值得加。

它的作用不是放 workflow，而是放 UI 展示信息：

- 显示名称
- 简短描述
- 默认提示词


In [8]:
print((skill_root / 'agents' / 'openai.yaml').read_text(encoding='utf-8'))


interface:
  display_name: "Weather"
  short_description: "Get current weather and forecasts without an API key"
  default_prompt: "Use Weather for current weather and short forecast questions. Prefer wttr.in first, and use Open-Meteo as a JSON-friendly fallback."



## 这一课最重要的正式开发原则

先把 Skill 做小，做准，做能用。

不要一开始就试图把它做成一个“大而全”的框架。

对这份天气 Skill 来说，第一版只要做到下面几点就够了：

1. 用户问天气时能匹配到这份 Skill
2. Skill 能提醒智能体先确认地点
3. Skill 能约束回答保持简洁
4. 文件结构保持最小且清晰


In [9]:
lesson_two_principles = [
    'small first',
    'workflow first',
    'no extra folders without need',
    'keep SKILL.md high-signal',
]

pprint(lesson_two_principles)


['small first',
 'workflow first',
 'no extra folders without need',
 'keep SKILL.md high-signal']


## 如果以后要把这份 Skill 继续做大，应该怎么扩展

第二版比较自然的扩展方向有两个：

1. 加 `references/`
   用来放地点格式、回答规范、天气字段说明

2. 加 `scripts/`
   用来做地点标准化或响应校验

也就是说，扩展应该来自真实需求，而不是预先堆目录。


In [10]:
future_extensions = {
    'references/': ['location_format.md', 'response_style.md'],
    'scripts/': ['normalize_location.py', 'validate_weather_response.py'],
}

pprint(future_extensions)


{'references/': ['location_format.md', 'response_style.md'],
 'scripts/': ['normalize_location.py', 'validate_weather_response.py']}


## 这份最小天气 Skill 在正式开发里应该怎么使用

你可以把它理解成一份“任务处理模板”。

当真实请求是：

- “北京今天什么天气？”
- “帮我看下上海这周末会不会下雨”

这份 Skill 的作用不是亲自生成天气数据，而是指导智能体：

1. 先确认地点是否明确
2. 再调用天气能力
3. 再按约定格式回答

也就是说，Skill 管 workflow，天气工具或天气能力管数据获取。


In [ ]:
real_usage_flow = [
    'user asks weather question',
    'skill matches because task is weather query',
    'agent checks location clarity',
    'agent uses weather capability/tool',
    'agent answers briefly and practically',
]

pprint(real_usage_flow)


## 这一课可以怎么验收

你学完这一课，至少应该能回答这 4 个问题：

1. 为什么这份天气 Skill 现在只需要两个文件？
2. 为什么现在不急着建 `references/`？
3. 为什么现在不急着建 `scripts/`？
4. 这份 Skill 和天气工具各自负责什么？


In [ ]:
lesson_two_check = [
    'Can explain why minimal structure is enough',
    'Can explain when references become necessary',
    'Can explain when scripts become necessary',
    'Can explain skill responsibility vs weather capability responsibility',
]

pprint(lesson_two_check)


## 当前阶段结论

你现在需要记住：

1. 第二课最重要的是学会“先做最小 Skill”
2. 一份最小 Skill，很多时候只需要 `SKILL.md`，外加推荐的 `agents/openai.yaml`
3. `references/` 和 `scripts/` 不是默认必备，而是后续按需求增长
4. 天气 Skill 的职责是定义处理天气请求的 workflow，不是自己提供天气数据
5. 正式开发时，目录扩展应该来自真实需求，而不是来自“看起来完整”

下一步建议：

- 继续第三课：给天气 Skill 增加 `references/`
- 或者继续第三课：给天气 Skill 增加一个地点标准化脚本
